# 05c — Hybrid Routing Pipeline
**Domain:** Healthcare / Multi-domain &nbsp;|&nbsp; **Time:** ~1.5 h  
**Key Concepts:** confidence threshold, cost/quality tradeoff, routing log

---
### Idea
```
patient_row → ML model → predict_proba → confidence score
    high confidence (>= threshold) → return ML prediction  (cheap, fast)
    low  confidence (< threshold)  → call LLM for explanation (expensive, slower)
```
Log every decision so you can audit the cost/quality tradeoff.


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
import pandas as pd, numpy as np
from xgboost import XGBClassifier
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# ── Rebuild heart model from 05b (or use synthetic data) ──────────────────
HEART_PATH = os.path.join("data", "heart_disease", "heart.csv")
if os.path.exists(HEART_PATH):
    heart_df = pd.read_csv(HEART_PATH)
else:
    rng = np.random.default_rng(42); n = 303
    heart_df = pd.DataFrame({
        "age": rng.integers(30,80,n), "sex": rng.integers(0,2,n),
        "cp": rng.integers(0,4,n), "trestbps": rng.integers(90,200,n),
        "chol": rng.integers(150,400,n), "fbs": rng.integers(0,2,n),
        "thalach": rng.integers(80,200,n), "exang": rng.integers(0,2,n),
        "oldpeak": rng.uniform(0,6,n).round(1), "target": rng.integers(0,2,n),
    })

feature_cols = [c for c in heart_df.columns if c != "target"]
X = heart_df[feature_cols].values
y = heart_df["target"].values

model = XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric="logloss")
model.fit(X, y)
print("✅ Heart model ready")


In [ ]:
show_exercise(
    "05c-1", "Hybrid router with threshold and cost tracking",
    """Write route_prediction(patient_row: pd.Series, threshold: float = 0.8) -> dict.
If max(predict_proba) >= threshold → route='ml', cost_usd=0.0
Else                               → route='llm', call LLM, cost_usd=0.001

Run on 50 patients. Save routing_log.csv with columns:
  patient_id, route, confidence, prediction, cost_usd""",
    hints=[
        "proba = model.predict_proba(row.values.reshape(1,-1))[0]",
        "confidence = max(proba); prediction = int(np.argmax(proba))",
        "pd.DataFrame(log).to_csv('routing_log.csv', index=False)",
    ],
    checks=[
        "route_prediction() returns dict with 'route', 'prediction', 'confidence', 'cost_usd'",
        "routing_log.csv saved with all required columns",
        "Both 'ml' and 'llm' routes used across 50 patients",
    ],
)


In [ ]:
LLM_COST = 0.001   # $ per LLM call (estimated)
ML_COST  = 0.000   # free

def route_prediction(patient_row: "pd.Series", threshold: float = 0.8) -> dict:
    """Route to ML or LLM based on model confidence."""
    # TODO: compute predict_proba → confidence, prediction
    # TODO: if confidence >= threshold: return ML result (cost_usd=0)
    # TODO: else: call LLM for plain-English explanation (cost_usd=LLM_COST)
    pass


# ── Run on 50 patients ─────────────────────────────────────────────────────
routing_log = []
for i in range(50):
    row = heart_df.iloc[i]
    try:
        result = route_prediction(row)
        routing_log.append({"patient_id": i, **result})
    except Exception as e:
        print(f"Patient {i}: {e}")

routing_df = pd.DataFrame(routing_log)
print(routing_df.head())


In [ ]:
if len(routing_df):
    routing_df.to_csv("routing_log.csv", index=False)
    print("✅ routing_log.csv saved")

ml_routes  = (routing_df["route"] == "ml" ).sum() if "route" in routing_df.columns else 0
llm_routes = (routing_df["route"] == "llm").sum() if "route" in routing_df.columns else 0
total_cost = routing_df["cost_usd"].sum()          if "cost_usd" in routing_df.columns else 0
print(f"ML: {ml_routes} | LLM: {llm_routes} | Total cost: ${total_cost:.3f}")

check([
    (len(routing_log) >= 50,                           f"{len(routing_log)} patients routed"),
    (os.path.exists("routing_log.csv"),                "routing_log.csv saved"),
    (ml_routes  > 0,                                   f"ML route used ({ml_routes} patients)"),
    (llm_routes > 0,                                   f"LLM route used ({llm_routes} patients)"),
    ("confidence" in routing_df.columns,               "'confidence' column present"),
    ("cost_usd"   in routing_df.columns,               "'cost_usd' column present"),
], exercise_id="05c-1")


In [ ]:
evaluate(route_prediction,
         "Hybrid router: ML if confidence>=threshold, else LLM. Save routing_log.csv with all fields.",
         exercise_id="05c-1")
progress()
